# Experiment 5: Randomized Benchmarking Protocol

### Aim:
Implement the Randomized Benchmarking Protocol to measure quantum gate fidelity.

---
### Dataset Required:
None. Benchmarking is performed on randomly sampled Clifford gates.

---
### Libraries Required & Purpose:
* `qiskit`: To generate Clifford gates and compute inverse operators.
* `qiskit_aer`: To simulate circuits under gate noise.
* `numpy`: To randomly select gates.

---


### Step 1: Import Qiskit, NumPy, and Simulator

In [ ]:
# Import necessary modules
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

### Step 2: Function to Create an RB Circuit

In [ ]:
# Single-qubit Clifford basis gates
GATES = ['x', 'y', 'z', 'h', 's']

def make_rb_circuit(length):
    qc = QuantumCircuit(1, 1)
    
    # 1. Apply 'length' random Clifford gates
    chosen = np.random.choice(GATES, size=length)
    for g in chosen:
        getattr(qc, g)(0)
        
    # 2. Compute and apply the exact inverse gate to return qubit to |0>
    inv = Operator(qc).adjoint().data
    qc.unitary(inv, [0])
    
    # 3. Measure qubit 0
    qc.measure(0, 0)
    return qc

# Show sample circuit of length 4
sample = make_rb_circuit(4)
print(sample.draw(output='text'))

### Step 3: Run Benchmarking with Simulated Noise

In [ ]:
# Create a simple noise model with 2% depolarizing error per gate
noise = NoiseModel()
noise.add_all_qubit_quantum_error(depolarizing_error(0.02, 1), ['x', 'y', 'z', 'h', 's', 'unitary'])
sim = AerSimulator(noise_model=noise)

# Test survival probability across sequence lengths
lengths = [1, 5, 10, 20]
print("Randomized Benchmarking Results:")
for m in lengths:
    circuit = make_rb_circuit(m)
    compiled = transpile(circuit, sim)
    counts = sim.run(compiled, shots=1024).result().get_counts()
    p0 = counts.get('0', 0) / 1024
    print(f"Sequence Length m = {m:2d} | Survival Probability P(|0>): {p0:.4f}")